In [0]:
import pyspark.sql.functions as F

In [0]:
RENAME_MAP = {
    'rank' : 'player_rank',
    'player' : 'player_id',
    'points' : 'current_points'
}

#Reading from Bronze

In [0]:
df = spark.table('tennislakehouse.bronze.rankings')

#Cleaning the data

In [0]:
#removing exact duplicate rows
df = df.distinct()

In [0]:
#casting the ranking date to an actual date
df = df.withColumn(
    'ranking_date',
    F.try_to_date(F.col("ranking_date").cast("string"), "yyyyMMdd")
)

In [0]:
#filtering out rows with null points
df = df.filter(~(F.col('points').isNull()))

#Renaming columns

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

#Writing to Silver

In [0]:
(
df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable('tennislakehouse.silver.rankings')
)